# Parallel Experiment Launcher — Local Test
Simulates what `train_job_parallel.sh` does, without SLURM.
Run cells top to bottom.

In [1]:
# ─── Configuration — edit these to match your setup ───────────────────────────
EXPERIMENTS  = [7135, 7136] #, 7137, 7138]
DATA_DIR     = '/home/sgomezro/scratch/metrics/Exp'
PYTHON_SCRIPT = 'main.py'
GPU_ID       = 0
LOG_BASE     = 'logs/parallel_tests'          # logs go to experiments/Exp_XXXX/
# ──────────────────────────────────────────────────────────────────────────────

In [10]:
import subprocess, os, time, threading
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets
import sys
print(sys.executable)

# Create log directories
for exp_id in EXPERIMENTS:
    Path(f'{LOG_BASE}/Exp_{exp_id}').mkdir(parents=True, exist_ok=True)
print('Log directories ready.')

/home/sgomezro/envs/merl_env/bin/python
Log directories ready.


In [11]:
# ─── GPU memory check (requires nvidia-smi) ───────────────────────────────────
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True, check=True
    )
    name, total, free = result.stdout.strip().split(', ')
    print(f'GPU : {name}')
    print(f'VRAM: {free} MB free / {total} MB total')
    print(f'Running {len(EXPERIMENTS)} experiments — ensure each fits in ~{int(free)//len(EXPERIMENTS)} MB')
except FileNotFoundError:
    print('nvidia-smi not found — skipping GPU check (CPU-only or driver not installed).')

GPU : NVIDIA H100 80GB HBM3
VRAM: 50738 MB free / 81559 MB total
Running 2 experiments — ensure each fits in ~25369 MB


In [12]:
# ─── Launch all experiments in parallel ───────────────────────────────────────
processes = {}   # exp_id -> Popen
status    = {}   # exp_id -> 'running' | 'done' | 'failed'

env = os.environ.copy()
env['OMP_NUM_THREADS'] = '2'
# MPS env vars (harmless locally if MPS isn't running)
env['CUDA_MPS_PIPE_DIRECTORY'] = '/tmp/nvidia-mps'
env['CUDA_MPS_LOG_DIRECTORY']  = '/tmp/nvidia-log'

for exp_id in EXPERIMENTS:
    stdout_path = f'{LOG_BASE}/Exp_{exp_id}/output.log'
    stderr_path = f'{LOG_BASE}/Exp_{exp_id}/error.log'
    cmd = ['python', PYTHON_SCRIPT,
           '-g', str(GPU_ID),
           '-e', str(exp_id),
           '-d', DATA_DIR]
    with open(stdout_path, 'w') as out, open(stderr_path, 'w') as err:
        proc = subprocess.Popen(cmd, stdout=out, stderr=err, env=env)
    processes[exp_id] = proc
    status[exp_id]    = 'running'
    print(f'Launched Exp {exp_id}  (PID {proc.pid})')
    time.sleep(1)   # stagger startup

print(f'\nAll {len(EXPERIMENTS)} experiments running.')

Launched Exp 7135  (PID 1021567)
Launched Exp 7136  (PID 1021568)

All 2 experiments running.


In [13]:
# ─── Live status monitor (polls every 5 s) ────────────────────────────────────
bars   = {e: widgets.IntProgress(value=0, min=0, max=100,
                                  description=f'Exp {e}',
                                  bar_style='info', layout=widgets.Layout(width='60%'))
          for e in EXPERIMENTS}
labels = {e: widgets.Label(value='running') for e in EXPERIMENTS}
rows   = [widgets.HBox([bars[e], labels[e]]) for e in EXPERIMENTS]
box    = widgets.VBox(rows)
display(box)

def monitor():
    while True:
        all_done = True
        for exp_id, proc in processes.items():
            if status[exp_id] == 'running':
                rc = proc.poll()
                if rc is None:
                    all_done = False
                elif rc == 0:
                    status[exp_id] = 'done'
                    bars[exp_id].value     = 100
                    bars[exp_id].bar_style = 'success'
                    labels[exp_id].value   = 'done ✓'
                else:
                    status[exp_id] = 'failed'
                    bars[exp_id].bar_style = 'danger'
                    labels[exp_id].value   = f'failed (rc={rc}) ✗'
            else:
                pass  # already settled
        if all_done:
            break
        time.sleep(5)

t = threading.Thread(target=monitor, daemon=True)
t.start()
print('Monitor running in background — check widgets above.')

Monitor running in background — check widgets above.


In [6]:
cmd = ['python', PYTHON_SCRIPT,
           '-g', str(GPU_ID),
           '-e', str(exp_id),
           '-d', DATA_DIR]

In [7]:
cmd

['python',
 'main.py',
 '-g',
 '0',
 '-e',
 '7136',
 '-d',
 '/home/sgomezro/scratch/metrics/Exp']

In [8]:
processes

{7135: <Popen: returncode: 1 args: ['python', 'main.py', '-g', '0', '-e', '7135', '...>,
 7136: <Popen: returncode: 1 args: ['python', 'main.py', '-g', '0', '-e', '7136', '...>}

In [9]:
cat experiments/Exp_7135/error.log

Traceback (most recent call last):
  File "/home/sgomezro/scratch/ai-ev-routing/main.py", line 6, in <module>
    import numpy as np
ModuleNotFoundError: No module named 'numpy'
